# perform_block_matching

Write a Python3 function `perform_block_matching(image_left, image_right, block_size, max_disparity)` which performs and returns the best disparity map using the block matching algorithm taught in CompSci773 lecture and tutorial. The function arguments 'image_left' and 'image_right' are a Python3 Numpy list of lists for the left image and the right image, taken from the two cameras. You may assume the left image and right image have been rectified. The function argument 'block_size' is the block size in the block matching algorithm, while 'max_disparity' is the maximum possible disparity value.

Your function should return a Python3 list of lists containing the computed disparity map with the same height and width as the left and right images. To do that, you need to implement the correct image border handling. In this question, you should extend the border by padding with boundary values of the input image (BorderBoundaryPadding). To do that, you may use our helper function 'reflect_make_border(image, offset)' to perform border padding and acquire the padded image. Function argument 'image' is your input image, and 'offset' is the padding width (offset = block_size // 2). You need to do that for both left and right images. Then, before you return the computed disparity map, you need to remove the padded regions to preserve the original image size.

For Block-Score (x, y, d) discussed in the lecture and tutorial, you should use NCC (Normalized Cross Correlation) to compute the cost between two blocks in the left and right images. You may use our helper function 'ncc_cost(block_left, block_right)' to compute NCC as well. The function arguments 'block_left' and 'block_right' are the left and right image windows, respectively.

Note: Do not round your results in any way!

Note: Do not perform any normalization on your disparity map!

Note: You may assume the Python module "Numpy" is available.

Note: You may download the following left and right images to test your code on your own code editor!

Click Here to Download Left Image, Click Here to Download Right Image

Note: You may also download the expected disparity map (in .npy format) below. To read the .npy file, use the following code: np.load('disparity_map.npy').

Click Here to Download Data



In [79]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [80]:
import numpy as np

In [81]:
# dx1 = 0.001
# dy1 = 0.001

# R1 = np.array([[0.7, -0.7, 0.05],
#               [-0.2, -0.06, 1.0],
#               [-0.7, -0.7, -0.1]])

# tx1 = 5.0
# ty1 = 10.0
# tz1 = 53.0
# f1 = 5.0
# sx1 = 1.0
# cx1 = 1500.0
# cy1 = 1125.0

# dx2 = 0.001
# dy2 = 0.001
# R2 = np.array([[0.7, -0.7, 0.04],
#                 [-0.3, -0.09, 1.0],
#                 [-0.7, -0.7, -0.2]])
# tx2 = 1.0
# ty2 = 2.0
# tz2 = 57.0
# f2 = 5.0
# sx2 = 1.0
# cx2 = 1500.0
# cy2 = 1125.0

In [83]:
# Calibration parameters
class H3LeftTsaiCali():
    def __init__(self):
        self.dx = 0.00155
        self.dy = 0.00155
        self.R = np.array([[ 0.69358669,  -0.7183941,  0.05336124],
                           [-0.13580093, -0.06134937,  0.98883485],
                           [-0.70710192, -0.69309163, -0.14011019]])
        self.tx = 2.0199458439516227
        self.ty = -9.952670170284382
        self.tz = 58.68943072204245
        self.f = 2.708689448793061
        self.sx = 1.0018491172850652
        self.cx = 1500
        self.cy = 1125

class H3RightTsaiCali():
    def __init__(self):
        self.dx = 0.00155
        self.dy = 0.00155
        self.R = np.array([[ 0.69863844, -0.71441153,  0.03899342],
                           [-0.14357909, -0.08914402,  0.98561574],
                           [-0.70066037, -0.69418882, -0.16485427]])
        self.tx = -1.6065117014923782
        self.ty = -8.054880535641873
        self.tz = 60.16429266232803
        self.f = 2.772940725368117
        self.sx = 1.0001142712275077
        self.cx = 1500
        self.cy = 1125



# # Homography matrices H1 (left image):
# [[ 0.6596, 0.0763, -3083.3251]
#  [-0.3281, 0.9803, -1670.0809]
#  [-0.0002,      0,    -0.6304]]
# # Homography matrices H2 (right image):
# [[ 0.6513, 0.093,  -3081.063],
#  [-0.3048, 0.977, -1660.0774],
#  [-0.0002,     0,    -0.6603]]

In [144]:
def rectification(h3LeftTsaiCali, h3RightTsaiCali):
    def t(cali):
        return np.array([cali.tx, cali.ty, cali.tz])
        
    def K(cali):
        return np.array([[cali.f * cali.sx / cali.dx, 0, cali.cx, 0],
                        [0, -cali.f / cali.dy, cali.cy, 0],
                        [0, 0, 1, 0]])
        
    def M(cali):
        last_row = np.array([[0, 0, 0, 1]])
        return np.vstack((np.hstack((cali.R, t(cali).reshape(-1, 1))), last_row))
        
    def P(cali):
        return K(cali) @ M(cali)
    
    c1 = -h3LeftTsaiCali.R.T @ t(h3LeftTsaiCali)
    c2 = -h3RightTsaiCali.R.T @ t(h3RightTsaiCali)
    c = c2 - c1
    # print("c:\n", c)
    
    Vx = c / (c @ c)**0.5
    Vy = np.cross(h3LeftTsaiCali.R[2, :], Vx)
    Vy = Vy / (Vy @ Vy)**0.5
    Vz = np.cross(Vx, Vy)

    R_new = np.stack((Vx, Vy, Vz))
    
    last_M_new_row = np.array([[0, 0, 0, 1]])
    M1_new = np.hstack((R_new, t(h3LeftTsaiCali).reshape(-1, 1)))
    M2_new = np.hstack((R_new, t(h3RightTsaiCali).reshape(-1, 1)))
    # add the last row [0, 0, 0, 1] to M1_new and M2_new
    M1_new = np.vstack((M1_new, last_M_new_row))
    M2_new = np.vstack((M2_new, last_M_new_row))
    
    K_new = (K(h3LeftTsaiCali) + K(h3RightTsaiCali)) / 2
    # print("K_new:\n", K_new)
    
    P1_new = K_new @ M1_new
    P2_new = K_new @ M2_new
    
    H1 = P1_new[:3, :3] @ np.linalg.inv(P(h3LeftTsaiCali)[ :3, :3])
    H2 = P2_new[:3, :3] @ np.linalg.inv(P(h3RightTsaiCali)[ :3, :3])
    
    return H1.tolist(), H2.tolist()

In [145]:
left_tsai_cali_H3 = H3LeftTsaiCali()
right_tsai_cali_H3 = H3RightTsaiCali()

H_l, H_r = rectification(left_tsai_cali_H3, right_tsai_cali_H3)

print("H_l:\n", H_l)
print("H_r:\n", H_r)

H_l:
 [[1.2324757849367223, 0.14264265012856617, -1192.1653124072], [0.10155194754714074, 1.0300390007133027, -251.71111158067683], [0.00019096496285494666, 2.2101649960393727e-05, 0.6303865945565431]]
H_r:
 [[1.2058226431429369, 0.10937934611743924, -1100.2392557687742], [0.11115445683407976, 0.9893224254253072, -174.45954716128608], [0.00018484668948472041, 5.455480664194311e-06, 0.6602745938889788]]
